# MaleCNS Connectome to 3D Humanoid Physics Simulation (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gakpey/MaleCNS_human/blob/main/notebooks/02_malecns_humanoid_simulation.ipynb)

This interactive notebook simulates a **3D Humanoid in a physics-based world** controlled by the **MaleCNS (*Drosophila melanogaster*) connectome**. Sensory stimuli propagate through MaleCNS interneurons to descending motor output neurons, driving human body movement.

> **Live Streaming Feature**: Watch the 3D simulation happen **live** inside your Google Colab browser cell without requiring any local GPU hardware!

## Step 1a: Clone Repository & Setup Directory
Detects the Colab runtime and clones the MaleCNS_human project if it isn't already present.

In [ ]:
import sys
import os
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("🚀 Google Colab detected!")
    repo_dir = Path("/content/MaleCNS_human")
    if not repo_dir.exists() and not Path("src/malecns").exists():
        print("📥 Cloning repository from GitHub...")
        !git clone https://github.com/Gakpey/MaleCNS_human.git /content/MaleCNS_human
    if repo_dir.exists():
        os.chdir(repo_dir)
        print(f"✔ Working directory: {os.getcwd()}")
else:
    print("💻 Running in Local Development Environment")

## Step 1b: Install Physics & Rendering Dependencies
Installs PyBullet, meshcat, imageio, and related packages. Live output is shown so you can monitor any metadata preparation steps in real-time.

In [ ]:
if IS_COLAB:
    # Install packages with live output (no -q flag so progress is visible)
    !pip install pybullet networkx meshcat imageio imageio-ffmpeg mediapy

# Add src/ to sys.path — bypasses editable install entirely for instant imports
possible_paths = [
    Path.cwd() / "src",
    Path.cwd().parent / "src",
    Path("/content/MaleCNS_human/src"),
    Path("/content/src")
]
for p in possible_paths:
    if p.exists() and str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))
        print(f"✔ Added {p.resolve()} to sys.path")

## Step 2: Initialize MaleCNS Connectome & Physics Engine

In [ ]:
from malecns.config import setup_environment
from malecns.utils import set_seed
from malecns.connectome import MaleCNSNetwork
from malecns.mapping import ConnectomeToHumanoidMapper
from malecns.simulation import HumanoidPhysicsSim
from malecns.render import LiveWebGLViewer, VideoRenderer

set_seed(42)
env_paths = setup_environment()

connectome = MaleCNSNetwork(num_sensory=16, num_interneurons=64, num_descending=16, seed=42)
print("🧠 MaleCNS Connectome Summary:", connectome.get_summary())

mapper = ConnectomeToHumanoidMapper(num_descending_neurons=16, gain=1.2, smoothing=0.15)
print(f"🦴 Mapped {len(mapper.get_joint_names())} 3D Humanoid Body Degrees of Freedom (DoFs)")

sim = HumanoidPhysicsSim(render_mode="DIRECT")

## Step 3: Setup Live WebGL 3D Viewer
Initializes a live Three.js 3D WebGL canvas directly inside your Colab output cell.

In [ ]:
live_viewer = LiveWebGLViewer(width=640, height=400)
live_viewer.init_colab_display()

## Step 4: Run Connectome-Driven Physics Simulation
Propagates sensory signals through MaleCNS neurons → maps firing rates to joint angles → steps the physics world → streams live 3D view → captures frames for MP4 export.

In [ ]:
import numpy as np
import time

sim_steps = 150
dt = 0.02
captured_frames = []

print("🎬 Starting Connectome-Driven Physics Simulation...")

for step in range(sim_steps):
    t = step * dt
    sensory_stimulus = np.zeros(16)
    sensory_stimulus[0:4] = 0.8 * np.sin(2.0 * np.pi * 0.5 * t) + 0.2
    sensory_stimulus[4:8] = 0.6 * np.cos(2.0 * np.pi * 0.8 * t)
    sensory_stimulus[8:12] = 0.9 * (1.0 if (step // 20) % 2 == 0 else 0.1)

    dn_firing_rates = connectome.step(sensory_stimulus, dt=dt)
    target_joint_angles = mapper.map_dn_to_joints(dn_firing_rates)

    sim.apply_joint_targets(target_joint_angles)
    sim.step()

    state = sim.get_state()
    pos = state["position"]

    if step % 2 == 0:
        live_viewer.update(position=pos, joint_angles=target_joint_angles, step=step)

    rgb_frame = sim.render_frame(width=640, height=480)
    captured_frames.append(rgb_frame)

    if (step + 1) % 30 == 0:
        print(f"   Step {step+1}/{sim_steps} — Humanoid Height: {pos[2]:.2f}m")

sim.close()
print("✅ Physics Simulation Complete!")

## Step 5: Render & Play Simulation MP4 Video
Compiles captured frames into an MP4 video with `yuv420p` format (web-compatible) and embeds a playback player in the cell output.

In [ ]:
output_video_path = env_paths["results"] / "malecns_humanoid_behavior.mp4"
VideoRenderer.save_video(captured_frames, str(output_video_path), fps=30)
VideoRenderer.display_video_in_colab(str(output_video_path), width=640)